In [ ]:
# Benjamin_ExtractCounties.py
# 4/24/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(5, 10)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries
  

# maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0


start_time=datetime.datetime.now()
event_time=start_time


process_name='Extract Benjamin State Counties '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInputStates='State_Id'
fileOutput="Benjamin_Counties"


fileStatesIn=DirMain+"Data\\"+fileInputStates+".csv"
fileOut=DirMain+"Data\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"Data\\"+fileOutput+".csv"

states_info_df=pd.read_csv(fileStatesIn)
state_id_list=states_info_df["state_id"]
state_name_list=states_info_df["state_name"]

my_header = {
  'ancestry-clientpath': 'findagrave-frontend',
  'apollographql-client-name': 'findagrave-cemetery-searchresults',
  'content-type': 'application/json',
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'
}

sleep_time0=1*60
county_df=pd.DataFrame([])


url_prefix="https://www.findagrave.com/orc/graphql?operationName=browse&variables=%7B%22parents%22%3A%5B%22top%22%2C%22continent_68%22%2C%22country_4%22%2C%22"
url_suffix="%22%5D%2C%22ignoreCemeteries%22%3Atrue%2C%22hasCemeteries%22%3Atrue%7D&extensions=%7B%22persistedQuery%22%3A%7B%22version%22%3A1%2C%22sha256Hash%22%3A%22bfb78b85f6be61da441f58ab1e538e6b36ba50f5437b177f7826953eb9ee02c8%22%7D%7D"

county_df_header=['state_name','state_id','county_id','county_name','county_lat','county_lon']

county_df=pd.DataFrame([])
state_count=-1
for state_id in state_id_list:
    state_count+=1
    # memorial_count_df=pd.DataFrame([])
    # cemetery_id_df=pd.DataFrame([])
    # cemetery_info_df=pd.DataFrame([])
    # cemetery_frame_pre_df=pd.DataFrame([])
    state_name=state_name_list.iloc[state_count]
    url=url_prefix+state_id+url_suffix
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code);continue
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 

    print(json.dumps(capture_json))
    input('capture_json')
    
    capture_json_df=pd.DataFrame(capture_json['data']['browse'])
    state_match=capture_json_df[capture_json_df['id'].str.contains(state_id)]
    state_match_location=state_match['locations']
    data = []
    for loc_list in state_match_location:
        for loc in loc_list:  # Each loc_list is a list of dicts
            county_id = loc.get('id')
            name = loc.get('names', [{}])[0].get('name', '')
            county_lat=loc.get('coordinates').get('lat', '')
            county_lon=loc.get('coordinates').get('lon', '')
            data.append((county_id, name,county_lat,county_lon))
    county_frame_df = pd.DataFrame(data, columns=['county_id', 'county_name','county_lat','county_lon'])
    county_count=len(county_frame_df)
    state_id_vector=pd.Series([state_id]).repeat(county_count).reset_index(drop=True)
    state_name_vector=pd.Series([state_name]).repeat(county_count).reset_index(drop=True)
    county_frame_df=pd.concat([state_name_vector,state_id_vector,county_frame_df], axis=1)
    county_frame_df.columns =county_df_header 
   
    county_df = county_df._append(county_frame_df, ignore_index = True) 
    end_time=datetime.datetime.now()
    processing_time=end_time-event_time
    event_time=end_time
    processing_time_seconds=int(processing_time.total_seconds())

    print(f"**** {process_name}  State: {state_name:<3} • County Count: {county_count:<3} •  Processing Time: {processing_time_seconds} Sec")
    sleep_time=random_sleep(state_count+1)       
    # time.sleep(sleep_time)

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}  • Processing Time: {processing_time_seconds} Sec")   

county_df.to_csv(fileOut,mode="w", index=False,header=True)
county_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


In [ ]:
state_match_location=state_match['locations']
list(state_match_location)
for loc_list in state_match_location:
    for loc in loc_list:  # Each loc_list is a list of dicts
        county_id = loc.get('id')
        name = loc.get('names', [{}])[0].get('name', '')
        county_lat=loc.get('coordinates').get('lat', '')
        county_lon=loc.get('coordinates').get('lat', '')
        data.append((county_id, name,county_lat,county_lon))
# county_frame_df = pd.DataFrame(data, columns=['county_id', 'county_name'])

In [207]:
collection_df=pd.json_normalize(capture_json['data']['browse'])

In [209]:
list(collection_df['locations'])

[[{'id': 'continent_74',
   'names': [{'name': 'Africa', '__typename': 'LocationName'},
    {'name': 'Africa', '__typename': 'LocationName'},
    {'name': 'Africa', '__typename': 'LocationName'},
    {'name': 'Africa', '__typename': 'LocationName'}],
   'locations': [{'cemetery': None,
     'city': None,
     'county': None,
     'state': None,
     'country': None,
     'continent': 'Africa',
     '__typename': 'LocationHierarchy'}],
   'coordinates': None,
   '__typename': 'Location'},
  {'id': 'continent_72',
   'names': [{'name': 'Antarctica', '__typename': 'LocationName'},
    {'name': 'Antarctica', '__typename': 'LocationName'},
    {'name': 'Antarctica', '__typename': 'LocationName'}],
   'locations': [{'cemetery': None,
     'city': None,
     'county': None,
     'state': None,
     'country': None,
     'continent': 'Antarctica',
     '__typename': 'LocationHierarchy'}],
   'coordinates': None,
   '__typename': 'Location'},
  {'id': 'continent_71',
   'names': [{'name': 'Asia